In [6]:
import numpy as np
from itertools import product
from math import ceil

# 初始方案的最优配送结果（万吨）
optimal_solution = np.array([
    [ 0.        , 30.175433907508,  9.06583095394121,  0.        ,  0.        ,  0.        ,
      0.        , 14.9595917197154,  0.        , 13.9291434188354,  0.        , 10.53     ,
      6.34      ],
    [31.5382204877364,  0.        ,  0.        ,  0.        ,  0.        , 43.3839807620296,
      0.        ,  0.        , 39.422623689384,  5.65517506085007,  0.        ,  0.        ,
      0.        ],
    [ 0.        ,  0.        ,  0.        , 36.9777190601011, 36.911715151657,  0.        ,
      0.        ,  0.        ,  0.        ,  0.725472406852631, 30.3850933813893,  0.        ,
      0.        ]
])*10000

# 车型信息
vehicle_info = {
    '小型车': {'最大运量': 10, '单位交通成本': 0.145, '单位油耗': 1.4},
    '中型车': {'最大运量': 20, '单位交通成本': 0.0975, '单位油耗': 3.6},
    '重型车': {'最大运量': 40, '单位交通成本': 0.045, '单位油耗': 6.8}
}

# 各仓库与配送目标间的公路距离（公里）
distances = np.array([
    [410.2, 63.7, 177.6, 562.2, 516.3, 275.4, 312.3, 162.0, 246.0, 138.0, 376.0, 135.9, 123.2],
    [80.8, 385.3, 530.2, 213.2, 167.3, 149.6, 439.1, 513.4, 168.2, 272.9, 70.6, 292.0, 271.7],
    [124.4, 428.6, 573.6, 150.3, 104.4, 222.5, 518.8, 556.5, 211.5, 315.9, 55.0, 335.1, 315.1]
])

# 仓库和城市名称映射
warehouse_names = ['仓库A(玉田)', '仓库B(辛集)', '仓库C(南宫)']
city_names = ['石家庄', '唐山', '秦皇岛', '邯郸', '邢台', '保定', '张家口', '承德', '沧州', '廊坊', '衡水', '北京', '天津']

# 按容量从大到小排序车型
vehicle_types = sorted(vehicle_info.keys(), key=lambda x: vehicle_info[x]['最大运量'], reverse=True)

def calculate_vehicle_cost(vehicle_type, distance, load):
    """计算单个车次的成本"""
    return (vehicle_info[vehicle_type]['单位交通成本'] * distance * load + 
            vehicle_info[vehicle_type]['单位油耗'] * distance)

def find_optimal_vehicle_combination(weight, distance):
    """使用动态规划找到最优车型组合"""
    # 预计算每种车型的单位成本 (成本/吨)
    unit_costs = {}
    for vehicle in vehicle_types:
        max_load = vehicle_info[vehicle]['最大运量']
        full_trip_cost = calculate_vehicle_cost(vehicle, distance, max_load)
        unit_costs[vehicle] = full_trip_cost / max_load
    
    # 按单位成本从低到高排序车型
    sorted_vehicles = sorted(vehicle_types, key=lambda x: unit_costs[x])
    
    # 初始化动态规划表
    dp = [float('inf')] * (ceil(weight) + 1)
    dp[0] = 0  # 基本情况：0吨需要0成本
    
    # 记录每种状态下使用的车型和数量
    choices = [[0, 0, 0] for _ in range(ceil(weight) + 1)]
    
    # 动态规划计算最小成本
    for w in range(1, ceil(weight) + 1):
        for vehicle in sorted_vehicles:
            capacity = vehicle_info[vehicle]['最大运量']
            cost_per_trip = calculate_vehicle_cost(vehicle, distance, capacity)
            
            # 尝试使用1辆该车型
            if w >= capacity:
                if dp[w - capacity] + cost_per_trip < dp[w]:
                    dp[w] = dp[w - capacity] + cost_per_trip
                    choices[w] = choices[w - capacity].copy()
                    vehicle_index = vehicle_types.index(vehicle)
                    choices[w][vehicle_index] += 1
            else:
                # 如果剩余重量不足一车，计算部分装载的成本
                partial_cost = calculate_vehicle_cost(vehicle, distance, w)
                if partial_cost < dp[w]:
                    dp[w] = partial_cost
                    choices[w] = [0, 0, 0]
                    vehicle_index = vehicle_types.index(vehicle)
                    choices[w][vehicle_index] += 1
    
    return tuple(choices[ceil(weight)]), dp[ceil(weight)]

print('车型优化后的配送方案（考虑所有可能的车型组合，选择最低成本）：')
total_trips = 0
total_cost = 0

# 计算总成本和总车次
for i, warehouse in enumerate(warehouse_names):
    print(f'\n{warehouse}:')
    for j, city in enumerate(city_names):
        if optimal_solution[i, j] > 0:
            print(f'  到{city} ({optimal_solution[i, j]:.2f}万吨):')
            weight = optimal_solution[i, j]
            distance = distances[i][j]
            
            # 找到最优车型组合
            best_combo, min_cost = find_optimal_vehicle_combination(weight, distance)
            
            # 输出最优方案
            total_cost += min_cost
            total_trips += sum(best_combo)
            
            for idx, vehicle in enumerate(vehicle_types):
                if best_combo[idx] > 0:
                    print(f'    {vehicle}: {best_combo[idx]}车次')
            
            print(f'    成本: {min_cost:.2f}元')

city_demands = np.array([31.538220487736385, 30.175433907507962, 9.065830953941212, 36.97771906010111, 36.911715151657, 43.38398076202959, 38.16967199494477, 14.959591719715412, 39.91295169443925, 20.309790886538106, 30.38509338138926, 10.53, 6.34])*10000
demand_satisfaction = np.sum(optimal_solution, axis=0) / city_demands

print(f'\n总运输车次: {total_trips}')
print(f'总运输成本: {total_cost:.2f}元')
# print('各地区需求满足率：', demand_satisfaction)
print('平均需求满足率：', np.mean(demand_satisfaction))

车型优化后的配送方案（考虑所有可能的车型组合，选择最低成本）：

仓库A(玉田):
  到唐山 (301754.34万吨):
    重型车: 7544车次
    成本: 4132739.75元
  到秦皇岛 (90658.31万吨):
    重型车: 2266车次
    中型车: 1车次
    成本: 3461966.12元
  到承德 (149595.92万吨):
    重型车: 3740车次
    成本: 5210538.84元
  到廊坊 (139291.43万吨):
    重型车: 3482车次
    小型车: 2车次
    成本: 4133064.12元
  到北京 (105300.00万吨):
    重型车: 2632车次
    中型车: 1车次
    成本: 3076877.93元
  到天津 (63400.00万吨):
    重型车: 1585车次
    成本: 1679339.20元

仓库B(辛集):
  到石家庄 (315382.20万吨):
    重型车: 7884车次
    中型车: 1车次
    小型车: 1车次
    成本: 5479030.63元
  到保定 (433839.81万吨):
    重型车: 10846车次
    成本: 13954029.76元
  到沧州 (394226.24万吨):
    重型车: 9855车次
    中型车: 1车次
    小型车: 1车次
    成本: 14256794.31元
  到廊坊 (56551.75万吨):
    重型车: 1414车次
    成本: 3318474.92元

仓库C(南宫):
  到邯郸 (369777.19万吨):
    重型车: 9244车次
    中型车: 1车次
    成本: 11949414.38元
  到邢台 (369117.15万吨):
    重型车: 9228车次
    成本: 8285258.12元
  到廊坊 (7254.72万吨):
    重型车: 181车次
    小型车: 2车次
    成本: 493301.54元
  到衡水 (303850.93万吨):
    重型车: 7596车次
    小型车: 2车次
    成本: 3593149.73元

总运输车次: 775